In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
from pathlib import Path

# Resolve project root regardless of where Jupyter was launched from
_here = Path(os.path.abspath(''))
PROJECT_ROOT = str(_here.parent if _here.name == 'notebooks' else _here)
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


import transformers
transformers.logging.set_verbosity_error()

import pandas as pd
import numpy as np
import pickle
from math import isnan
from tqdm import tqdm
from dataclasses import dataclass
from typing import Optional
import matplotlib.pyplot as plt

ALPHABET = 'ACDEFGHIKLMNPQRSTVWY-'

ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:
from utils.config import get_default_config, get_model_configs
from SaProtdG import SaProtdG, SaProtdG_predict

In [ ]:
def get_pdb(pdb_code=""):
  if pdb_code is None or pdb_code == "":
    raise ValueError("Please provide a PDB code (e.g. '1lze'), a local file path, or a UniProt ID.")
  elif os.path.isfile(pdb_code):
    return pdb_code
  elif len(pdb_code) == 4:
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    return f"{pdb_code}.pdb"
  else:
    os.system(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{pdb_code}-F1-model_v3.pdb")
    return f"AF-{pdb_code}-F1-model_v3.pdb"

## 1. wild-type prediction

In [ ]:
cfg = get_default_config()
cfg.testing.ddg_scanning = False

PDB_NAME = 'nanobody_1zvh'
CHAIN_ID = 'A'
WEIGHT_FILES = [
    "saprotdg_weights/SaProtdG_weights_augmented_1_lora.ckpt",
    "saprotdg_weights/SaProtdG_weights_augmented_2_lora.ckpt",
    "saprotdg_weights/SaProtdG_weights_augmented_3_lora.ckpt",
]

pdb_path = os.path.join(PROJECT_ROOT, "examples", "nanobody_1zvh.cif")
predictions = []
for weight_file in WEIGHT_FILES:
    model = SaProtdG(weight_file, cfg)
    _, pred_dg_avg, combined_seq = SaProtdG_predict(model, pdb_path, CHAIN_ID)
    predictions.append(pred_dg_avg[0])

avg_pred = sum(predictions) / len(predictions)
std_pred = (sum((x - avg_pred) ** 2 for x in predictions) / len(predictions)) ** 0.5
print(f"🧬 {PDB_NAME} Predicted ΔG (kcal/mol): {avg_pred:.2f} ± {std_pred:.2f}")

## 2. Mutational scanning

In [ ]:
cfg = get_default_config()
cfg.testing.ddg_scanning = True

PDB_NAME = 'nanobody_1zvh'
CHAIN_ID = 'A'
WEIGHT_FILES = [
    "saprotdg_weights/SaProtdG_weights_augmented_1_lora.ckpt",
    "saprotdg_weights/SaProtdG_weights_augmented_2_lora.ckpt",
    "saprotdg_weights/SaProtdG_weights_augmented_3_lora.ckpt",
]

pdb_path = os.path.join(PROJECT_ROOT, "examples", "nanobody_1zvh.cif")
all_scaled_ddg = []
for weight_file in WEIGHT_FILES:
    model = SaProtdG(weight_file, cfg)
    _, scaled_ddg, combined_seq = SaProtdG_predict(model, pdb_path, CHAIN_ID, ddg_scanning=True)
    all_scaled_ddg.append(scaled_ddg)

scaled_pred_mutant_ddg = torch.stack(all_scaled_ddg).mean(dim=0)

In [ ]:
mean_results = np.mean(scaled_pred_mutant_ddg.detach().cpu().numpy(), axis=-1).T[0]
height, width = mean_results.shape
fig_width = min(max(width/4, 8), 25)
fig_height = min(max(height/4, 3), 8)
plt.figure(figsize=(fig_width, fig_height))

vmin = mean_results.min()
vmax = mean_results.max()
divergence = max(abs(vmin), abs(vmax))

plt.imshow(mean_results,
           aspect='auto',
           cmap='bwr',
           vmin=-divergence,
           vmax=divergence)

L = mean_results.shape[1]
plt.xticks(ticks=np.arange(L), labels=np.arange(1, L + 1), rotation=90)
plt.yticks(ticks=range(len(ALPHABET)), labels=ALPHABET)
plt.ylabel("Amino Acids")
plt.title("Mutational Scanning {}".format(PDB_NAME))
plt.colorbar()
plt.show()